# Reset Notebook Configurations

In [ ]:
# Restart the Runtime (Hard Reset)
# This is the most effective way to completely clear RAM and disk cache.
import os
os.kill(os.getpid(), 9)

In [ ]:
# Delete variables
%reset -f

# Clear CUDA cache (only for deep learning examples)
import torch
torch.cuda.empty_cache()

# Clear garbage
import gc
gc.collect()

30

In [ ]:
!rm -rf /content/*
!rm -rf ~/.cache/huggingface

In [ ]:
!df -h       # Disk usage
print("="*100)
print("="*100)
!nvidia-smi  # GPU usage
print("="*100)
print("="*100)
!free -h     # RAM usage

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   65G  43% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G     0  5.7G   0% /dev/shm
/dev/root       2.0G  1.2G  750M  62% /usr/sbin/docker-init
/dev/sda1       119G   72G   48G  61% /opt/bin/.nvidia
tmpfs           6.4G  7.6M  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
Fri Nov 14 04:26:33 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|              

# Code Run

In [1]:
# # Install uv package manager
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.7/22.7 MB 89.9 MB/s eta 0:00:00:00:0100:01


In [2]:
# ============================================================================
# STEP 1: Install Dependencies
# ============================================================================
print("Installing qwen-tts package...")
!uv pip install -q qwen-tts soundfile

# Optional: Install Flash Attention 2 for better performance (requires compatible GPU)
# Note: This may take a few minutes
print("Installing flash-attention (optional, for better performance)...")
!uv pip install -q flash-attn --no-build-isolation

Installing qwen-tts package...
Installing flash-attention (optional, for better performance)...


In [3]:
# ============================================================================
# STEP 2: Import Libraries
# ============================================================================
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel
from IPython.display import Audio, display

print("Libraries imported successfully!")


    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    


Libraries imported successfully!


In [4]:
# ============================================================================
# STEP 3: Load the Model
# ============================================================================
print("Loading Qwen3-TTS model...")
print("This may take a few minutes on first run as it downloads the model weights...")

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0" if torch.cuda.is_available() else "cpu",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2" if torch.cuda.is_available() else "eager",
)

print("Model loaded successfully!")
print(f"Using device: {'CUDA (GPU)' if torch.cuda.is_available() else 'CPU'}")

Loading Qwen3-TTS model...
This may take a few minutes on first run as it downloads the model weights...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Model loaded successfully!
Using device: CUDA (GPU)


In [13]:
# Load reference audio and its transcript
ref_audio = "https://raw.githubusercontent.com/mcikalmerdeka/nlp-learning/main/qwen-tts-experiment/voice_clone/Mika_Growup_2.ogg.mp3"
ref_text  = "あははっ☆今のままじゃ満足できないんだ？じゃあ、もうちょっとだけ期待に応えちゃおっかな。"

ref_audio_2 = "https://raw.githubusercontent.com/mcikalmerdeka/nlp-learning/main/qwen-tts-experiment/voice_clone/Mika_Battle_In_1.ogg.mp3"
ref_text_2 = "それじゃ、始めよっか。"

In [16]:
# Generate voice clone (one time usage with same reference audio)
wavs, sr = model.generate_voice_clone(
    text="さあ、コーディングセッションを始めましょう。もう終わりましたか？まずはVSCodeアプリケーションを開いてみましょう。",
    language="Japanese",
    ref_audio=ref_audio,
    ref_text=ref_text,
)
sf.write("output_voice_clone.wav", wavs[0], sr)

# Display the generated clone audio
print("\nGenerated clone audio:")
display(Audio(wavs[0], rate=sr))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.



Generated clone audio:


In [17]:
# Generate voice clone (multiple usage with same reference prompt)
prompt_items = model.create_voice_clone_prompt(
    ## Using multiple reference audio and texts
    ## Note: The number of reference and generated have to be the same (will return ValueError, ex: Batch size mismatch: prompt=2, text=3)
    ## I thik that the first reference will be use for the first generated audio, etc
    ref_audio=[ref_audio, ref_audio_2], 
    ref_text=[ref_text, ref_text_2],
    x_vector_only_mode=False,
)

first_text = "それは素晴らしいと思います！はい、ぜひその選択肢を選ぶべきです。"
second_text = "LangChainは、大規模言語モデル（LLM）を活用したアプリケーションの開発を簡素化するために設計されたオープンソースのオーケストレーションフレームワークです。LLMを外部データソースやツールに接続するための標準インターフェースを提供し、開発者が複雑でコンテキストアウェアなAIエージェントやワークフローを構築できるようにします。"

wavs, sr = model.generate_voice_clone(
    text=[first_text, second_text],
    language=["Japanese", "Japanese"],
    voice_clone_prompt=prompt_items,
)
sf.write("output_voice_clone_2.wav", wavs[0], sr)
sf.write("output_voice_clone_3.wav", wavs[1], sr)

# Display the generated clone audio
print("\nGenerated clone audio:")
display(Audio(wavs[0], rate=sr))
display(Audio(wavs[1], rate=sr))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.



Generated clone audio:
